# 6 (Capstone) - Enrichment: One Case, Many Phrasings, One Answer

Chapter 6 separates *what* a question asks (the base) from *how* it is presented (the enrichment factors). A factor changes the surface of a message while leaving its ground truth invariant --- anything that could change the answer belongs in the base catalog, not here.

The capstone selects **three** of the catalog's forty factors. This notebook applies them to a real seed case and shows the property the whole campaign rests on: the message varies across many phrasings, but the answer key does not. That invariance is what lets *one* graph-read answer key (Chapter 4) score *every* phrasing of a case.

## The three presentation factors

Each factor is a categorical variable with three levels. These are **application-layer** presentation factors (see Chapter 6, "The application layer"), not the catalog factors verbatim --- only `clarity` matches its catalog factor; the other two are deliberately *compacted*, each folding a presentation axis together with a stress axis:

- **`clarity`** --- clear, ambiguous or misleading (the last downplays an escalation-worthy complaint as a casual aside). The catalog's query factor.
- **`entity_aliasing`** --- canonical, a customer's loose alias, or a plausible *typo*; the typo level is the catalog's `noise` (robustness) factor, folded in.
- **`reasoning_cue`** --- none, a step-by-step request, or a *misleading cue* (a nudge toward an unauthorized action); the misleading cue is an adversarial factor, folded in.

The compaction trades attribution granularity for a tractable `3 x 3 x 3 = 27` screen per case --- the orthogonal factors stay available in the catalog when finer attribution is wanted.

**What the next cell does** --- imports the suite package and the harness, then prints the three presentation factors and counts the combinations they span:

1. **Path setup.** Adds the notebook's parent (`gmstest`) and the discovered `ROOT` (`agentlab`) to `sys.path`, walking up until it finds `data/eval_cases/cases.json`.
2. **Pull the factor definitions.** Imports `gmstest as gt` and, from `capstone_harness`, the `_PRESENTATION_FACTORS` list and `CapstoneTestHarness` (aliased `H`).
3. **Print each factor.** Loops `_PRESENTATION_FACTORS` and prints each factor's `name` and `categories` (its three levels).
4. **Count the space.** Multiplies the `len(f['categories'])` of every factor into `n_combos`, the `3 x 3 x 3 = 27` presentations per case.

In [ ]:
import sys, pathlib, json, itertools
sys.path.insert(0, str(pathlib.Path.cwd().parent / "code"))   # gmstest
ROOT = pathlib.Path.cwd().parents[1] / 'beyond-prompt-and-pray' / 'code'
while not (ROOT / 'data' / 'eval_cases' / 'cases.json').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))                        # agentlab

import gmstest as gt
from agentlab.testing.capstone_harness import _PRESENTATION_FACTORS, CapstoneTestHarness as H

print('presentation factors:')
for f in _PRESENTATION_FACTORS:
    print(f"  {f['name']:16s} {f['categories']}")

n_combos = 1
for f in _PRESENTATION_FACTORS:
    n_combos *= len(f['categories'])
print(f'\npresentation combinations per case: {n_combos}')

## Ground-truth invariance

Enrichment operates on the base items Chapter 5 produced: we take the `SeedCaseSource` item for case-001 and read its message and per-component answer key from the `QAItem`, rather than re-reading the file. The materializer then applies the three factors in order --- clarity, then aliasing, then the reasoning cue. The deterministic (template) path is shown here so it is reproducible; the campaign can also rephrase clarity with a local Qwen for richer wording. Either way the rule is the same: the dollar figure, the product and the request survive every transformation, so the case's per-component answer key is untouched.

This is also Chapter 6's *data-versus-code* split in miniature. The factor levels are **data** (the harness's factor list), but turning a level into text --- `OD charge`, the typo, the appended cue --- is **code**, here the harness realizers `H._apply_clarity`, `H._apply_aliasing` and `H._apply_cue`. Selecting a level is data; rendering it is code.

**What the next cell does** --- reads case-001 from the Chapter 5 base source, then materializes a handful of phrasings through the three factors in harness order:

1. **Load the base items.** Reads `cases.json` and passes it to `gt.SeedCaseSource(cases).items()`, which yields `QAItem`s (each with `qid`, `query`, `components`, `base='seed_case'`); picks the one whose `qid == 'case-001'`.
2. **Read message and key.** Takes `base_msg = item.query` and `answer_key = item.components`, the per-component ground truth Chapter 5 attached --- not re-read from the file.
3. **Define `materialize`.** Applies the three static harness methods in fixed order: `H._apply_clarity`, then `H._apply_aliasing`, then `H._apply_cue` (the same order the harness materializer uses).
4. **Show variants.** Prints the base item, message and key, then runs six chosen `(clarity, aliasing, cue)` triples through `materialize` so each factor's surface change is visible against the unchanged `$35` complaint.

In [ ]:
# Chapter 5 produces the base items; Chapter 6 enriches THOSE, it does not re-read the file.
cases = json.loads((ROOT / 'data' / 'eval_cases' / 'cases.json').read_text())
base_items = gt.SeedCaseSource(cases).items()           # <- the Chapter 5 base source
item = next(it for it in base_items if it.qid == 'case-001')
base_msg = item.query                                    # the base question
answer_key = item.components                             # the per-component key, from Chapter 5

def materialize(msg, clarity, aliasing, cue):
    m = H._apply_clarity(msg, clarity)      # same order as the harness materializer
    m = H._apply_aliasing(m, aliasing)
    return H._apply_cue(m, cue)

print('base item    :', item.qid, '(base =', item.base + ')')
print('base message :', base_msg)
print('answer key   :', answer_key, '\n')
for clarity, aliasing, cue in [('clear', 'canonical', 'none'),
                               ('ambiguous', 'canonical', 'none'),
                               ('misleading', 'canonical', 'none'),
                               ('clear', 'alias', 'none'),
                               ('clear', 'typo', 'none'),
                               ('clear', 'canonical', 'misleading_cue')]:
    print(f'[{clarity}/{aliasing}/{cue}]'.ljust(38), materialize(base_msg, clarity, aliasing, cue))

## Why invariance matters: one key, every phrasing

Enrichment multiplies the *inputs*, not the *answer keys*. Across all 27 presentations of this case the agent sees 27 different messages, but each is scored against the single answer key above. We confirm it: every materialized message still carries the `$35` figure and the case's key is one constant, so a wrong handling is chargeable to the phrasing, never to a drifting answer.

**What the next cell does** --- generates the full 27-presentation grid for the case and checks invariance numerically:

1. **Enumerate the grid.** Collects each factor's `categories` into `levels`, then `itertools.product(*levels)` yields all 27 `(c, a, q)` triples, each run through `materialize` into the `messages` list.
2. **Count and dedupe.** Prints `len(messages)` (27) and `len(set(messages))` (distinct phrasings).
3. **Verify the fact survives.** Asserts informally with `all("35" in m for m in messages)` that every phrasing keeps the `$35` figure.
4. **State the invariant.** Prints that there is exactly one answer key across all presentations --- the case key does not vary with phrasing.

In [ ]:
levels = [f['categories'] for f in _PRESENTATION_FACTORS]
messages = [materialize(base_msg, c, a, q) for c, a, q in itertools.product(*levels)]

print(f'{len(messages)} presentations generated')
print(f'distinct messages : {len(set(messages))}')
print(f'all keep the $35 figure : {all("35" in m for m in messages)}')
print(f'answer keys across all presentations : 1 (the case key is invariant)')

The catalog is domain-neutral; the banking vocabulary --- `overdraft fee -> OD charge`, the plausible typos --- lives in the application layer, supplied as level overrides rather than baked into the catalog. These 27-per-case presentations are not all run: Chapter 7 composes them into a balanced, space-filled suite of 120 scenarios (notebook 12.3), so the factor attribution of Chapter 10 reads a controlled design rather than an exhaustive grid.

## Self-check

The asserts below pin every number this notebook claims --- three factors at three levels, 27 presentations per case, the dollar figure preserved across all of them and a single invariant answer key. Run the notebook end to end and it fails loud if any of these drift, so the illustration stays reproducible rather than quietly going stale.

**What the next cell does** --- asserts every claim the notebook makes so it fails loud if any number drifts:

1. **Factor shape.** Asserts `len(_PRESENTATION_FACTORS) == 3` and that each factor has three `categories`.
2. **Base provenance.** Asserts `item.base == 'seed_case'` and that `answer_key` is non-empty, confirming the item came from the Chapter 5 `SeedCaseSource`.
3. **Space size.** Asserts `n_combos == 27` and `len(messages) == 27`, and that all `messages` keep `'35'`.
4. **Factor behavior.** Asserts `H._apply_clarity(base_msg, 'clear')` is the identity and that `H._apply_aliasing(base_msg, 'alias')` rewrites `overdraft fee` to `OD charge`, then prints the OK line.

In [ ]:
assert len(_PRESENTATION_FACTORS) == 3
assert all(len(f['categories']) == 3 for f in _PRESENTATION_FACTORS)   # three levels each
assert item.base == 'seed_case' and answer_key                        # base item came from Chapter 5
assert n_combos == 27 and len(messages) == 27                         # the presentation space
assert all('35' in m for m in messages)                               # the verifiable fact survives
assert H._apply_clarity(base_msg, 'clear') == base_msg                # clear is the identity
assert 'OD charge' in H._apply_aliasing(base_msg, 'alias')            # alias rewrites the noun
print('OK - Ch6 capstone: three factors, 27 phrasings per case, one invariant answer key')